In [1]:
from march.module import DMC
from march.ops.grid import pc_to_voxel_grid

import torch
import open3d as o3d
import open3d.core as o3c
import numpy as np
import plotly.graph_objects as go
import util

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Object

In [2]:
sphere_mesh = o3d.t.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=20)

print("# vertices:", len(sphere_mesh.vertex.positions))
print("# triangles:", len(sphere_mesh.triangle.indices))

# vertices: 762
# triangles: 1520


# Voxel Grid

In [3]:
num_sampling_points = 100000
device = 'cuda' if torch.cuda.is_available() else 'cpu'

points = sphere_mesh.sample_points_uniformly(num_sampling_points)
points = torch.from_numpy(points.point.positions.cpu().numpy()).float().to(device)

print(f"pc: {points.shape}, {points.dtype}")

pc: torch.Size([100000, 3]), torch.float32


In [4]:
voxel_grid_res = 8

grid_vertices, cubes = pc_to_voxel_grid(
    points=points,
    res_x=voxel_grid_res,
    res_y=voxel_grid_res,
    res_z=voxel_grid_res,
    k=1,
    num_keep=3,
    rmi_x=1, rmi_y=1, rmi_z=1,
    rma_x=1, rma_y=1, rma_z=1,
)

print(f"grid_vertices: {grid_vertices.shape}, cubes: {cubes.shape}")
print(f"grid_vertices: {grid_vertices.dtype}, cubes: {cubes.dtype}")
print(f"grid_vertices min: {grid_vertices.min()}, grid_vertices max: {grid_vertices.max()}")

grid_vertices: torch.Size([729, 3]), cubes: torch.Size([512, 8])
grid_vertices: torch.float32, cubes: torch.int32
grid_vertices min: -0.9996141791343689, grid_vertices max: 0.9997615218162537


# Sign Distance

In [5]:
scene = o3d.t.geometry.RaycastingScene()
scene.add_triangles(sphere_mesh)

iso = 0.0

In [6]:
queries = o3c.Tensor(grid_vertices.cpu().numpy(), dtype=o3c.float32)
print("queries:", queries.shape)

queries: SizeVector[729, 3]


In [7]:
distances = scene.compute_signed_distance(queries)

print("distances:", distances.shape)
print("distances (min, max):", distances.min().item(), distances.max().item())

distances: SizeVector[729]
distances (min, max): -0.9937715530395508 0.7313382029533386


In [8]:
print(type(distances))

<class 'open3d.cuda.pybind.core.Tensor'>


In [9]:
fig = go.Figure()

# Create color array: red if value > iso, blue otherwise
colors = ['red' if v > iso else 'blue' for v in distances.cpu().numpy()]

grid_vertices_np = grid_vertices.cpu().numpy()
print("grid_vertices_np:", grid_vertices_np.shape, grid_vertices_np.dtype)

# Add grid points as scatter plot
fig.add_trace(go.Scatter3d(
    x=grid_vertices_np[:, 0],
    y=grid_vertices_np[:, 1],
    z=grid_vertices_np[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    name='Grid Points'
))

grid_vertices_np: (729, 3) float32


# Reconstruction

In [10]:
distances = torch.from_numpy(distances.cpu().numpy()).to(device)

In [11]:
print(f"grid_vertices dtype: {grid_vertices.dtype}, device: {grid_vertices.device}")
print(f"cubes dtype: {cubes.dtype}, device: {cubes.device}")
print(f"distances dtype: {distances.dtype}, device: {distances.device}")

grid_vertices dtype: torch.float32, device: cuda:0
cubes dtype: torch.int32, device: cuda:0
distances dtype: torch.float32, device: cuda:0


In [12]:
mesh_vertices = torch.from_numpy(sphere_mesh.vertex.positions.cpu().numpy()).to(device)
mesh_faces = torch.from_numpy(sphere_mesh.triangle.indices.cpu().numpy()).to(device)

print(f"mesh_vertices dtype: {mesh_vertices.dtype}, device: {mesh_vertices.device}")
print(f"mesh_faces dtype: {mesh_faces.dtype}, device: {mesh_faces.device}")

mesh_vertices dtype: torch.float32, device: cuda:0
mesh_faces dtype: torch.int64, device: cuda:0


In [13]:
mc = DMC()

vertices, triangles = mc(
    grid_vertices=grid_vertices, 
    cubes=cubes, 
    values=distances, 
    iso=iso
)

In [14]:
print(f"# vertices: {vertices.shape[0]}, # triangles: {triangles.shape[0]}")
print(f"vertices dtype: {vertices.dtype}, device: {vertices.device}")
print(f"triangles dtype: {triangles.dtype}, device: {triangles.device}")

# vertices: 288, # triangles: 560
vertices dtype: torch.float32, device: cuda:0
triangles dtype: torch.int64, device: cuda:0


In [15]:
import kaolin as kal

final_mesh = kal.rep.SurfaceMesh(vertices=vertices, faces=triangles)

In [16]:
util.visualize_mesh(final_mesh, plot_normals=True, plot_wireframe=True)

In [17]:
# Export to OBJ

# o3d_mesh = o3d.geometry.TriangleMesh()
# o3d_mesh.vertices = o3d.utility.Vector3dVector(verts_numpy)
# o3d_mesh.triangles = o3d.utility.Vector3iVector(faces_numpy)
# o3d.io.write_triangle_mesh("marching_cubes_sphere.obj", o3d_mesh)